In [2]:
import sys
import os
import shutil
from pathlib import Path
import gdown

In [3]:
# Me aseguro de que el directorio raíz del proyecto esté en el sys.path
project_root = Path(os.path.abspath("")).parent  # Ajusta según sea necesario

# Añado el directorio raíz al sys.path si no está ya presente
if project_root not in sys.path:
    sys.path.append(str(project_root))

from src.config import raw_data_dir, interim_data_dir, load_config


Current working directory: /home/jorge/development/WasteImageReconstructionDL/notebooks
Loading configuration from /home/jorge/development/WasteImageReconstructionDL/src/config.yaml


In [4]:
print(project_root)

/home/jorge/development/WasteImageReconstructionDL


In [5]:
# Cargo la configuración 
config = load_config()
file_id = config['data']['google_drive_file_id']
destination = raw_data_dir() / 'reduced_dataset.zip'

Loading configuration from /home/jorge/development/WasteImageReconstructionDL/src/config.yaml


In [6]:
# Descargo el archivo desde Google Drive
if not raw_data_dir().exists():
    raw_data_dir().mkdir(parents=True)

In [7]:
print(file_id)

1LxVQ6na5njNsWWwYD_d2LswPoaRsor9C


In [8]:
# Convertimos el objeto Path a una cadena de texto
destination_str = str(destination)

url = f'https://drive.google.com/uc?id={file_id}'

gdown.download(url, destination_str, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1LxVQ6na5njNsWWwYD_d2LswPoaRsor9C
From (redirected): https://drive.google.com/uc?id=1LxVQ6na5njNsWWwYD_d2LswPoaRsor9C&confirm=t&uuid=9bb6a64d-cb5d-42d0-8df9-0a45e73abc7a
To: /home/jorge/development/WasteImageReconstructionDL/data/raw/reduced_dataset.zip
100%|██████████| 9.01G/9.01G [02:55<00:00, 51.4MB/s]


'/home/jorge/development/WasteImageReconstructionDL/data/raw/reduced_dataset.zip'

In [9]:
# Descomprimimos el archivo si es necesario
import zipfile
with zipfile.ZipFile(destination_str, 'r') as zip_ref:
    zip_ref.extractall(raw_data_dir())

In [10]:
# Elimino el archivo zip después de descomprimirlo
os.remove(destination_str)

### Copio las imagenes a un solo directorio

In [11]:
# Defino los directorios de entrada y salida
extracted_data_dir = raw_data_dir() / 'reduced_dataset' / 'data'
destination_dir = interim_data_dir() / 'dataset'

In [12]:
# Creo el directorio de destino si no existe
if not destination_dir.exists():
    destination_dir.mkdir(parents=True)

In [13]:
# Recorremos todos los archivos y subdirectorios en extracted_data_dir
for root, dirs, files in os.walk(extracted_data_dir):
    # Construir la ruta relativa a partir de extracted_data_dir
    relative_path = os.path.relpath(root, extracted_data_dir)
    
    # Creamos el directorio correspondiente en destination_dir
    destination_subdir = destination_dir / relative_path
    if not destination_subdir.exists():
        destination_subdir.mkdir(parents=True)
    
    # Copiaamos cada archivo al directorio correspondiente en destination_dir
    for file in files:
        source_file = Path(root) / file
        destination_file = destination_subdir / file
        shutil.copy2(source_file, destination_file)

print(f"Archivos copiados de {extracted_data_dir} a {destination_dir} manteniendo la estructura.")

Archivos copiados de /home/jorge/development/WasteImageReconstructionDL/data/raw/reduced_dataset/data a /home/jorge/development/WasteImageReconstructionDL/data/interim/dataset manteniendo la estructura.
